In [ ]:
!pip install roboflow torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 113.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [ ]:
import os
import torch
import torchvision
import numpy as np
from PIL import Image

from torch.utils.data import DataLoader
from torchvision.datasets import CocoDetection
import torchvision.transforms as T

In [ ]:
from roboflow import Roboflow


rf = Roboflow(api_key="FubcSJVDHEnV0FhKPMWy")
project = rf.workspace("nguyns-workspace-th4ak").project("dengue-fever-detection")
version = project.version(1)

dataset = version.download("coco")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to dengue-fever-detection-1 in coco:: 100%|██████████| 3193/3193 [00:00<00:00, 3382.18it/s]


In [ ]:
from torchvision.datasets import CocoDetection
import torch
import os
from PIL import Image

class RoboflowDetection(CocoDetection):
    def __init__(self, img_folder, ann_file, transforms=None):

        super().__init__(img_folder, ann_file, transforms=None)
        self.custom_transforms = transforms


        original_ids = list(self.ids)
        self.ids = []

        for img_id in original_ids:

            ann_ids = self.coco.getAnnIds(imgIds=img_id, iscrowd=None)
            if ann_ids:
                self.ids.append(img_id)

    def __getitem__(self, idx):

        img, target_coco = super().__getitem__(idx)

        boxes = []
        labels = []

        for obj in target_coco:
            x, y, w, h = obj['bbox']

            boxes.append([x, y, x+w, y+h])
            labels.append(1)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)


        coco_img_id = self.ids[idx]

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([coco_img_id])         }


        if self.custom_transforms:
            img = self.custom_transforms(img)

        return img, target

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
import torchvision.transforms as T

transforms = T.Compose([
    T.ToTensor()
])

In [ ]:
import os
from torch.utils.data import DataLoader

train_dataset = RoboflowDetection(
    img_folder=os.path.join(dataset.location, "train"),
    ann_file=os.path.join(dataset.location, "train/_annotations.coco.json"),
    transforms=transforms
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


In [ ]:
images, targets = next(iter(train_loader))

print(images[0].shape)
print(targets[0])

torch.Size([3, 512, 512])
{'boxes': tensor([[107.0000,  15.0000, 510.3480, 451.6440],
        [  6.0000, 350.0000, 115.8220, 444.5560]]), 'labels': tensor([1, 1]), 'image_id': tensor([1807])}


In [ ]:
print(train_dataset.coco.dataset['categories'])

[{'id': 0, 'name': 'objects', 'supercategory': 'none'}, {'id': 1, 'name': 'dengue-rash', 'supercategory': 'objects'}]


In [ ]:
import torchvision

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 161MB/s]


In [ ]:
num_classes = 2

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
    in_features, num_classes
)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.0005)

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for images, targets in loader:


        images = [img for img, t in zip(images, targets) if len(t["boxes"]) > 0]
        targets = [t for t in targets if len(t["boxes"]) > 0]

        if len(images) == 0:
            continue

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    return total_loss

In [ ]:
for epoch in range(3):
    loss = train_one_epoch(model, train_loader, optimizer)
    print(loss)

95.73518490791321
87.62099979072809
84.72246763110161


In [ ]:
model.eval()

images, _ = next(iter(train_loader))
images = [img.to(device) for img in images]

outputs = model(images)

print(outputs[0])

{'boxes': tensor([[ 82.3080,  22.7385, 428.2707, 486.1168],
        [ 46.8645,  10.6943, 512.0000, 307.0387],
        [ 31.6944, 256.0580, 473.2884, 494.5140],
        [  0.0000,  58.1132,  79.0108, 332.5330],
        [106.8673, 355.5880, 362.8729, 492.1895],
        [ 84.8087,   7.8471, 509.8509, 172.4722],
        [  0.0000,   0.0000, 104.2788, 252.6616],
        [279.2198, 236.3473, 411.8872, 476.3429],
        [194.1489, 351.0556, 439.4789, 488.8983],
        [  0.0000,   0.0000,  68.8771,  98.3861],
        [  0.0000, 177.6528,  80.8515, 284.8161],
        [288.4336,  17.1721, 429.7823, 272.3470],
        [  0.0000, 102.3802,  79.1428, 210.7532],
        [123.6665, 241.0649, 260.1375, 480.6299],
        [217.4166,  28.3134, 455.7886, 148.2708],
        [  5.4875,  12.1914,  74.2885, 195.4260],
        [  0.0000, 166.8591,  66.2118, 365.7957],
        [  0.0000, 220.5736,  72.1193, 322.1391],
        [ 99.1520, 309.9111, 300.0102, 463.4745],
        [259.4955, 318.8924, 473.0822, 4

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
model.eval()

total_boxes = 0

for images, targets in train_loader:
    images = [img.to(device) for img in images]
    outputs = model(images)

    total_boxes += len(outputs[0]["boxes"])
    break

print("Predicted boxes:", total_boxes)

Predicted boxes: 30
